# Токенизация: считаем токены и оцениваем модель

In [2]:
import tiktoken

text = "Студенты любят практику и хотят понять, как LLM видит текст."

try:
    encoding = tiktoken.encoding_for_model("gpt-5.4-mini")
except KeyError:
    encoding = tiktoken.get_encoding("o200k_base")

token_ids = encoding.encode(text)

print(token_ids) # -> [44098, 3388, ... , 60350, 13]
print(len(token_ids)) # -> 17

[44098, 3388, 29814, 171190, 32399, 4374, 816, 141869, 102970, 11, 6220, 451, 19641, 11431, 1108, 60350, 13]
17


In [3]:
tokens = [encoding.decode([token_id]) for token_id in token_ids]

print(tokens)

['Ст', 'уд', 'енты', ' любят', ' практи', 'ку', ' и', ' хотят', ' понять', ',', ' как', ' L', 'LM', ' вид', 'ит', ' текст', '.']


### Мини-функция для подсчёта токенов

In [5]:
import tiktoken

def get_openai_encoding(model_name: str):
    try:
        return tiktoken.encoding_for_model(model_name)
    except KeyError:
        return tiktoken.get_encoding("o200k_base")


def count_tokens(text: str, model_name: str = "gpt-5") -> int:
    encoding = get_openai_encoding(model_name)
    token_ids = encoding.encode(text)
    return len(token_ids)

In [6]:
text = "Студенты любят практику и хотят понять, как LLM видит текст."

count_tokens(text) # -> 17

17

In [7]:
def show_tokens(text: str, model_name: str = "gpt-5") -> list[str]:
    encoding = get_openai_encoding(model_name)
    token_ids = encoding.encode(text)
    return [encoding.decode([token_id]) for token_id in token_ids]

show_tokens("Привет, мир!") # -> ['Пр', 'ивет', ',', ' мир', '!']

['Пр', 'ивет', ',', ' мир', '!']

### Посмотрим на разные типы текста

In [11]:
texts = {
    "english": "Tokenization helps estimate the context size before sending a request.",
    "russian": "Токенизация помогает заранее оценить размер контекста.",
    "mixed": "Пользователь asked the модель to summarize 3 JSON файла.",
    "code": "def calculate_price(price, discount): return price * (1 - discount)",
    "numbers": "order_id=984120, sku_id=100239991, date=2026-05-10",
    "emoji": "🛠 ℹ ⚖ 📐 🔑 🏧.",
}

for name, text in texts.items():
    print(
        name,
        "characters:",
        len(text),
        "tokens:",
        count_tokens(text),
    )
    print(show_tokens(text))


english characters: 70 tokens: 12
['Token', 'ization', ' helps', ' estimate', ' the', ' context', ' size', ' before', ' sending', ' a', ' request', '.']
russian characters: 54 tokens: 13
['Т', 'ок', 'ен', 'изация', ' помогает', ' заранее', ' оцен', 'ить', ' размер', ' конт', 'ек', 'ста', '.']
mixed characters: 56 tokens: 13
['П', 'ольз', 'ователь', ' asked', ' the', ' модель', ' to', ' summarize', ' ', '3', ' JSON', ' файла', '.']
code characters: 67 tokens: 15
['def', ' calculate', '_price', '(price', ',', ' discount', '):', ' return', ' price', ' *', ' (', '1', ' -', ' discount', ')']
numbers characters: 50 tokens: 21
['order', '_id', '=', '984', '120', ',', ' sku', '_id', '=', '100', '239', '991', ',', ' date', '=', '202', '6', '-', '05', '-', '10']
emoji characters: 12 tokens: 15
['�', '�', '�', ' �', '�', ' �', '�', ' �', '�', ' �', '�', ' �', '�', '�', '.']


## Токенизация через HuggingFace🤗 tokenizer

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

/Users/diman/git_projects/praktikum-ai-engineer/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
text = "Студенты любят практику и хотят понять, как LLM видит текст."

encoded = tokenizer(text, add_special_tokens=False)

token_ids = encoded["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(token_ids)

print(token_ids) # -> [857, 1500, 4466, ..., 464, 29889]
print(tokens) # -> ['▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', ..., '▁L', 'LM', '▁ви', 'ди', 'т', '▁тек', 'ст', '.']
print(len(token_ids)) # -> 27

[857, 1500, 4466, 3327, 6331, 29975, 10691, 4460, 9054, 1382, 606, 6785, 6566, 29932, 733, 1200, 1413, 29892, 5413, 365, 26369, 2196, 956, 29932, 23567, 464, 29889]
['▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '▁и', '▁хо', 'тя', 'т', '▁по', 'ня', 'ть', ',', '▁как', '▁L', 'LM', '▁ви', 'ди', 'т', '▁тек', 'ст', '.']
27


In [15]:
### Мини-функции для HuggingFace🤗 tokenizer

In [16]:
def count_hf_tokens(text: str) -> int:
    encoded = tokenizer(text, add_special_tokens=False)
    return len(encoded["input_ids"])

def show_hf_tokens(text: str) -> list[str]:
    encoded = tokenizer(text, add_special_tokens=False)
    token_ids = encoded["input_ids"]
    return tokenizer.convert_ids_to_tokens(token_ids)

text = "Студенты любят практику❤️"

print(show_hf_tokens(text))
print(count_hf_tokens(text))

['▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '<0xE2>', '<0x9D>', '<0xA4>', '️']
14


In [17]:
token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
decoded_text = tokenizer.decode(token_ids)

print(decoded_text) # -> Студенты любят практику❤️

Студенты любят практику❤️


### Что изменится, если добавить специальные токены

In [19]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

text = "Студенты любят практику❤️"

encoded_without_special = tokenizer(text, add_special_tokens=False)
encoded_with_special = tokenizer(text, add_special_tokens=True)

tokens_without_special = tokenizer.convert_ids_to_tokens(
    encoded_without_special["input_ids"]
)
tokens_with_special = tokenizer.convert_ids_to_tokens(
    encoded_with_special["input_ids"]
)

print("Без специальных токенов:")
print(tokens_without_special)
print(len(tokens_without_special))

print("Со специальными токенами:")
print(tokens_with_special)
print(len(tokens_with_special))

Без специальных токенов:
['▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '<0xE2>', '<0x9D>', '<0xA4>', '️']
14
Со специальными токенами:
['<s>', '▁С', 'ту', 'ден', 'ты', '▁лю', 'б', 'ят', '▁пра', 'кти', 'ку', '<0xE2>', '<0x9D>', '<0xA4>', '️']
15


In [20]:
print(tokenizer.special_tokens_map)

{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}


### Более наглядный пример: BERT-токенизатор

In [22]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

text = "Студенты любят практику"

encoded_without_special = bert_tokenizer(text, add_special_tokens=False)
encoded_with_special = bert_tokenizer(text, add_special_tokens=True)

tokens_without_special = bert_tokenizer.convert_ids_to_tokens(
    encoded_without_special["input_ids"]
)
tokens_with_special = bert_tokenizer.convert_ids_to_tokens(
    encoded_with_special["input_ids"]
)

print("Без специальных токенов:")
print(tokens_without_special)
print(len(tokens_without_special))

print("Со специальными токенами:")
print(tokens_with_special)
print(len(tokens_with_special))

Без специальных токенов:
['С', '##ту', '##дент', '##ы', 'л', '##ю', '##б', '##ят', 'пра', '##кти', '##ку']
11
Со специальными токенами:
['[CLS]', 'С', '##ту', '##дент', '##ы', 'л', '##ю', '##б', '##ят', 'пра', '##кти', '##ку', '[SEP]']
13


In [23]:
first_text = "Студенты любят практику"
second_text = "Токенизация помогает считать стоимость"

encoded_pair = bert_tokenizer(
    first_text,
    second_text,
    add_special_tokens=True,
)

pair_tokens = bert_tokenizer.convert_ids_to_tokens(
    encoded_pair["input_ids"]
)

print(pair_tokens)

['[CLS]', 'С', '##ту', '##дент', '##ы', 'л', '##ю', '##б', '##ят', 'пра', '##кти', '##ку', '[SEP]', 'То', '##кен', '##изация', 'пом', '##ога', '##ет', 'с', '##читать', 'сто', '##имость', '[SEP]']


### Специальные токены и служебные обозначения — не одно и то же

```
Специальные токены — это отдельные токены со своими id в словаре. Они добавляются для технических целей:  
<s> — начало последовательности;
</s> — конец последовательности;
[CLS] — начало входа в BERT-подобных моделях;
[SEP] — разделитель;
<pad> или [PAD] — выравнивание длины;
<unk> или [UNK] — неизвестный фрагмент.
Служебные обозначения внутри токенов — это способ показать, как токен связан с соседними частями текста:  
▁ часто показывает пробел или начало слова;
## показывает продолжение предыдущего слова;  
На первый взгляд всё это выглядит как технический мусор. Но это не мусор: так токенизатор сохраняет структуру входа и информацию, нужную для восстановления текста.
```

## Сравним два токенизатора на одинаковых текстах

In [27]:
texts = {
    "english": "Tokenization helps estimate the context size before sending a request.",
    "russian": "Токенизация помогает заранее оценить размер контекста.",
    "mixed": "Пользователь asked the модель to summarize 3 JSON файла.",
    "code": "def calculate_price(price, discount): return price * (1 - discount)",
    "numbers": "order_id=984120, sku_id=100239991, date=2026-05-10",
    "emoji": "🛠 ℹ ⚖ 📐 🔑 🏧.",
}

In [28]:
import pandas as pd

rows = []

for text_type, text in texts.items():
    gpt_tokens = count_tokens(text)
    llama_tokens = count_hf_tokens(text)

    rows.append(
        {
            "text_type": text_type,
            "characters": len(text),
            "gpt_tokens": gpt_tokens,
            "llama_tokens": llama_tokens,
            "difference": llama_tokens - gpt_tokens,
        }
    )

comparison_df = pd.DataFrame(rows)

print(comparison_df)

  text_type  characters  gpt_tokens  llama_tokens  difference
0   english          70          12            12           0
1   russian          54          13            21           8
2     mixed          56          13            17           4
3      code          67          15            19           4
4   numbers          50          21            38          17
5     emoji          12          15            29          14


# Задание: 
```
Вы выбираете модель для обработки пользовательских обращений. 

Напишите код, который:
Посчитает стоимость одного запроса для GPT-модели.
Посчитает стоимость одного запроса для LLaMA-подобной модели.
Посчитает стоимость 10 000 запросов для каждой модели.
Выведет, какая модель проходит по бюджету.
```

In [31]:
# Один и тот же запрос уже токенизировали двумя токенизаторами:

gpt_input_tokens = 78
llama_input_tokens = 112

# Ожидаемая длина ответа:
expected_output_tokens = 500

# Цены моделей за 1 миллион токенов:
gpt_input_price = 0.75
gpt_output_price = 4.50

llama_input_price = 0.60
llama_output_price = 0.60

# Бюджет на 10 000 запросов:
budget = 20
requests_count = 10_000

In [40]:
def estimate_cost(n_in: int, n_out: int, c_in: float, c_out: float):
    return (n_in / 1_000_000) * c_in + (n_out / 1_000_000) * c_out 

gpt_cost = estimate_cost(gpt_input_tokens, expected_output_tokens, gpt_input_price, gpt_output_price) * 10_000
gpt_in_budget = gpt_cost <= budget
print(gpt_cost, gpt_in_budget)

llama_cost = estimate_cost(llama_input_tokens, expected_output_tokens, llama_input_price, llama_output_price) * 10_000
llama_in_budget = llama_cost <= budget
print(llama_cost, llama_in_budget)

if gpt_in_budget and llama_in_budget:
    print("Обе модели проходят по бюджету")
elif gpt_in_budget:
    print("По бюджету проходит GPT")
elif llama_in_budget:
    print("По бюджету проходит LLaMA")
else:
    print("Ни одна модель не проходит по бюджету") 

23.085 False
3.6719999999999997 True
По бюджету проходит LLaMA
